# Phase 0 - Environment Check

**CSGY-6513 Big Data Final Project: The Language of Complaints**

Goal: confirm Colab Pro can boot PySpark in local mode, hit the SODA API for some sample 311 data, and print the schema. Once this notebook runs end-to-end on a fresh kernel, Phase 0 is done and we move to ingestion.

**Run this in Google Colab Pro with a High-RAM runtime.**

## Cell 1 - Mount Drive

Drive gives us a place to write parquet files that survives across Colab sessions. Without this, every restart starts from scratch.

In [ ]:
# mount drive at /content/drive so we can persist intermediate parquets across colab sessions
from google.colab import drive
drive.mount('/content/drive')

# create our project root on drive if it doesnt exist yet
import os
os.makedirs('/content/drive/MyDrive/cs6513', exist_ok=True)
print('drive mounted at /content/drive')
print('project root: /content/drive/MyDrive/cs6513')

## Cell 2 - Clone the GitHub repo

We work out of the GitHub repo so we can `import src.*` for the reusable modules. After Phase 0 we'll routinely pull the latest at the start of each session.

In [ ]:
# project repo
REPO_URL = 'https://github.com/george-gideon-S/cs-gy-6513-big-data-311-nlp.git'

# clone (idempotent - if folder exists, just pull latest)
import os, subprocess
if os.path.isdir('/content/project/.git'):
    subprocess.run(['git', '-C', '/content/project', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, '/content/project'], check=True)

# add the project root to PYTHONPATH so `from src.foo import bar` just works
import sys
if '/content/project' not in sys.path:
    sys.path.insert(0, '/content/project')

print('repo cloned. files:')
subprocess.run(['ls', '/content/project'])

## Cell 3 - Install pinned dependencies

All deps live in `requirements.txt`. Same file Streamlit Cloud uses, so what works here works in deployment too.

In [ ]:
# install everything quietly. -q hides the giant pip output that bloats the notebook file
!pip install -r /content/project/requirements-train.txt -q
print('dependencies installed')

## Cell 4 - Java 11 setup

Colab ships Java 17 by default, which has a netty loopback bug with PySpark on Linux. Java 11 doesnt. We install it and point JAVA_HOME at it before anything Spark-related runs.

In [ ]:
# install openjdk 11 - this takes about 30 sec
!apt-get install -y openjdk-11-jre-headless > /dev/null 2>&1

# point JAVA_HOME at it. pyspark uses this env var to find java.
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

# verify
!java -version

## Cell 5 - Boot the SparkSession

Local mode with all cores, Delta Lake configured. If this prints a Spark version, the rest of the project is unblocked.

In [ ]:
# import our spark setup helper
from src.spark_setup import get_spark
spark = get_spark(app_name='phase0-env-check')

print('spark version:', spark.version)
print('master:', spark.sparkContext.master)
print('default parallelism:', spark.sparkContext.defaultParallelism)

## Cell 6 - Pull 10K rows from SODA API and print the schema

Sanity check that we can actually reach NYC Open Data and ingest into Spark. If this works, Phase 0 is done.

**Optional but recommended:** add a free SODA app token. https://data.cityofnewyork.us/profile/edit/developer_settings -> create token -> add it to Colab Secrets (left sidebar key icon) named `SODA_APP_TOKEN`. Without it, anonymous calls throttle around 1000 requests/hour, which is fine for Phase 0 but tight for Phase 1's 2M-row pull.

In [ ]:
# pull the soda token from colab secrets if available
import os
try:
    from google.colab import userdata
    token = userdata.get('SODA_APP_TOKEN')
    if token:
        os.environ['SODA_APP_TOKEN'] = token
        print('soda app token detected - good, throttling avoided')
    else:
        print('no soda app token - anonymous mode (fine for 10k rows)')
except Exception:
    print('no soda app token - anonymous mode (fine for 10k rows)')

# pull 10k rows from the 2020+ dataset
from src.config import SODA_2020_PLUS
from src.ingest import fetch_soda_paginated, normalize_columns

pdf = fetch_soda_paginated(SODA_2020_PLUS, target_rows=10_000, page_size=10_000)
print(f'\nfetched {len(pdf):,} rows from soda')
print('columns received:', len(pdf.columns))

In [ ]:
# convert to spark and run the column normalizer to confirm it works
sdf = spark.createDataFrame(pdf)
sdf_norm = normalize_columns(sdf)

print('normalized schema:')
sdf_norm.printSchema()
print(f'\nrow count: {sdf_norm.count():,}')
print('\nsample rows:')
sdf_norm.select('unique_key', 'created_date', 'agency', 'problem', 'borough').show(5, truncate=40)

## Phase 0 - Done when

- Spark version 3.5.x printed in Cell 5.
- Cell 6 prints a normalized schema with `unique_key`, `created_date`, `problem`, `problem_detail`, `borough` columns.
- Sample rows show actual NYC data.

If all three checks pass, save this notebook back to GitHub (File menu -> Save a copy in GitHub) and move on to `01_ingest.ipynb` for Phase 1.